In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
import torch
from peft import PeftModel
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
from gpu_monitor import *
model_name = "LORA_THE_BEST"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
import time
import pandas as pd
from tqdm import tqdm

device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
prompt = "Мой кариотип 46,XY,der(21;21)(q10;q10),+21. Что это значит?"
inputs = tokenizer([prompt], return_tensors="pt").to(device)   # <-- вот так
model.past_key_values = None
streamer = TextStreamer(tokenizer, skip_prompt=True)
start_time = time.perf_counter()
output_ids = model.generate(
    **inputs,
    #streamer=streamer,
    max_new_tokens=1600,
    early_stopping=True,
    use_cache=False, 
    #no_repeat_ngram_size=2,
    #eos_token_id=tokenizer.eos_token_id,
)
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
end_time = time.perf_counter()

print("Ответ модели:")
print(answer)
print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Ответ модели:
Мой кариотип 46,XY,der(21;21)(q10;q10),+21. Что это значит? Что изменения произошли в клетках организма? Что это значит для здоровья человека? Что это значит для риска рождения ребёнка с синдромом Дауна? Что это значит для семейного консультирования? Что это значит для планирования будущих беременностей? Что это значит для наблюдения пациента? Что это значит для оценки прогноза? Что это значит для поддержки родителей? Что это значит для образования фонограммы риска? Что это значит для интерпретации генетического консультирования? Что это значит для обследования родителей? Что это значит для оценки наследования? Что это значит для генетического консультирования родителей? Что это значит для планирования беременности с учетом инвазивной диагностики? Что это значит для наблюдения материале? Что это значит для оценки риска повторного рождения ребёнка с синдромом Дауна? Что это значит для поддержки материале? Что это значит для образования фонограммы риска повторного рождения?

In [3]:


def transformers_ai_answer(prompt):
    monitor = GPUMonitor(device_id=0)
    monitor.start()
    
    start_time = time.perf_counter()
    
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    model.past_key_values = None
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    start_time = time.perf_counter()
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1600,
        early_stopping=True,
        use_cache=False, 

    )
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    end_time = time.perf_counter()

    
    avg_power, max_power, avg_util = monitor.stop()
    response_time = end_time - start_time
    input_len = inputs['input_ids'].shape[1]
    output_len = output_ids.shape[1]
    completion_tokens = output_len - input_len
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    tokens_per_sec_per_watt = tokens_per_second / avg_power if avg_power > 0 else 0
    
    return {
        'answer': answer,
        'response_time': response_time,
        'completion_tokens': completion_tokens,
        'tokens_per_second': tokens_per_second,
        'avg_power_watts': avg_power,
        'max_power_watts': max_power,
        'avg_gpu_util': avg_util,
        'tokens_per_sec_per_watt': tokens_per_sec_per_watt
    }

In [4]:
transformers_ai_answer("Мой кариотип 47,XX,+i(12)(p10). Что это значит?")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'answer': 'Мой кариотип 47,XX,+i(12)(p10). Что это значит? Что ещё i(12p)?\nКариотип: 47,XX,+i(12)(p10)\nТип: изохромосома короткого плеча 12 (i(12p))\nГруппа: синдром Паллаистера–Киллиана\n\n1. Суть\nДополнительная изохромосома 12p → тетрасомия 12p (обычно мозаичная).\n\n2. Клиника\n• выраженная задержка развития\n• гипотония\n• характерные лицевые особенности\n• врождённые пороки\n\n3. Особенности\nЧасто не выявляется в крови, требуется исследование фибробластов.\n\n4. Заключение\ni(12p) ассоциирована с синдромом Паллаистера–Киллиана, тяжесть зависит от уровня мозаицизма.',
 'response_time': 4.6624695059999794,
 'completion_tokens': 205,
 'tokens_per_second': 43.96811598149697,
 'avg_power_watts': 469.1891914893617,
 'max_power_watts': 480.912,
 'avg_gpu_util': 94.23404255319149,
 'tokens_per_sec_per_watt': 0.09371084581451594}

In [5]:
df = pd.read_excel("Q_F_T.xlsx")
df

,Promts,References
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн..."
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом..."
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ..."
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес..."
...,...,...
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома..."
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а..."
103,"Мой кариотип 46,XX,del(3)(q29q29). Что это зна...","Кариотип: 46,XX,del(3)(q29q29)\nТип хромосомно..."


In [8]:
results = []
for prompt in tqdm(df['Promts']):
     res = transformers_ai_answer(prompt)
     results.append(res)

100%|██████████| 105/105 [2:21:48<00:00, 81.04s/it]


In [9]:
answers = [item['answer'] for item in results]
response_times = [item['response_time'] for item in results]
completion_tokens = [item['completion_tokens'] for item in results]
tokens_per_second = [item['tokens_per_second'] for item in results]
avg_power = [item['avg_power_watts'] for item in results]
max_power = [item['max_power_watts'] for item in results]
avg_gpu_util = [item['avg_gpu_util'] for item in results]
tps_per_watt = [item['tokens_per_sec_per_watt'] for item in results]

In [10]:
df["Answers"] = answers
df["Response times"] = response_times
df["Completion tokens"] = completion_tokens
df["Tokens per second"] = tokens_per_second
df["Avg Power (W)"] = avg_power
df["Max Power (W)"] = max_power
df["Avg GPU Util (%)"] = avg_gpu_util
df["Tokens/s per Watt"] = tps_per_watt

In [11]:
df.to_excel("TRANSFORMERS_TEST.xlsx", index = False)
df

,Promts,References,Answers,Response times,Completion tokens,Tokens per second,Avg Power (W),Max Power (W),Avg GPU Util (%),Tokens/s per Watt
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",63.952476,1096,17.137726,462.040781,480.725,96.757812,0.037091
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн...","Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...",125.636507,1600,12.735152,469.658362,480.040,98.307325,0.027116
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом...","Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...",24.763738,610,24.632792,469.559060,481.335,96.661290,0.052459
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ...","Мой кариотип 49,XXXXY. Что это значит? Что изм...",124.728043,1600,12.827909,469.779755,480.182,98.165998,0.027306
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес...","Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...",125.969802,1600,12.701457,469.815464,480.369,98.277778,0.027035
...,...,...,...,...,...,...,...,...,...,...
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",126.655504,1600,12.632692,469.988679,476.595,98.305687,0.026879
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома...","Мой кариотип 47,XX,+i(12)(p10). Что это значит...",126.187605,1600,12.679534,469.856376,479.451,98.318542,0.026986
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а...","Мой кариотип 46,XX,del(8)(p22). Что это значит...",126.024630,1600,12.695931,469.766775,479.585,98.277778,0.027026
103,"Мой кариотип 46,XX,del(3)(q29q29). Что это зна...","Кариотип: 46,XX,del(3)(q29q29)\nТип хромосомно...","Мой кариотип 46,XX,del(3)(q29q29). Что это зна...",7.377847,280,37.951450,468.152189,479.577,94.540541,0.081066
